# Classificador de fraude

As features `produto`, `valor` e `fraude` foram produzidas pelo job Spark `resumo-fraude` e publicadas no Feature Store do projeto `fraude`. Este notebook lê esse contrato, treina um classificador com os dados do bucket `dados` e grava o artefato em `s3://modelos/fraude`.

O modelo é tabular. Ele sobe num Inference Server em CPU. O Llama continua no vLLM, usando a fatia de GPU que já está reservada.

O mesmo fluxo, com as etapas separadas, está no pipeline **Fraude: dados, treino e deployment**.


In [ ]:
import json, ssl, urllib.request

token = open("/var/run/secrets/kubernetes.io/serviceaccount/token").read()
url = "https://feast-fraude-registry-rest.repositorio.svc.cluster.local/api/v1/features?project=fraude"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token})
with urllib.request.urlopen(req, context=ssl._create_unverified_context(), timeout=30) as resp:
    corpo = json.load(resp)

nomes = sorted({
    item.get("name")
    for item in corpo.get("features", [])
    if (item.get("featureView") or item.get("feature_view")) == "transacao_features"
})
print(nomes)
assert {"produto", "valor", "fraude"} <= set(nomes)


In [ ]:
import os
import boto3
import pandas as pd

cliente = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    config=boto3.session.Config(s3={"addressing_style": "path"}),
)
cliente.download_file("dados", "fraude/treino.csv", "treino.csv")
treino = pd.read_csv("treino.csv")
print(len(treino))
treino[treino["id"] < 2000]


In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

partes = train_test_split(treino, test_size=0.25, random_state=7, stratify=treino["fraude"])
treino_df, teste_df = partes
modelo = Pipeline([
    ("prep", ColumnTransformer([
        ("produto", OneHotEncoder(handle_unknown="ignore"), ["produto"]),
        ("valor", StandardScaler(), ["valor"]),
    ])),
    ("clf", LogisticRegression(max_iter=500)),
])
modelo.fit(treino_df[["produto", "valor"]], treino_df["fraude"])
previsto = modelo.predict(teste_df[["produto", "valor"]])
metricas = {
    "acuracia": round(float(accuracy_score(teste_df["fraude"], previsto)), 4),
    "precisao": round(float(precision_score(teste_df["fraude"], previsto)), 4),
    "recall": round(float(recall_score(teste_df["fraude"], previsto)), 4),
    "f1": round(float(f1_score(teste_df["fraude"], previsto)), 4),
    "amostras_teste": int(len(teste_df)),
}
print(metricas)
assert metricas["f1"] >= 0.8
joblib.dump(modelo, "model.joblib")
json.dump(metricas, open("metricas.json", "w"))


In [ ]:
cliente.upload_file("model.joblib", "modelos", "fraude/model.joblib")
cliente.upload_file("metricas.json", "modelos", "fraude/metricas.json")
print("s3://modelos/fraude")


## Inference Server

O bloco abaixo cria ou atualiza o `InferenceService` `fraude-classifier`. A rota pública já existe no gateway `openshift-ai-inference` e usa o header `X-Model: fraude-classifier`.


In [ ]:
import time
import urllib.error

namespace = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read()
host = os.environ.get("KUBERNETES_SERVICE_HOST", "kubernetes.default.svc")
base = f"https://{host}"
headers = {"Authorization": "Bearer " + token, "Content-Type": "application/json"}
contexto = ssl._create_unverified_context()
caminho = f"/apis/serving.kserve.io/v1beta1/namespaces/{namespace}/inferenceservices/fraude-classifier"
manifesto = {
    "apiVersion": "serving.kserve.io/v1beta1",
    "kind": "InferenceService",
    "metadata": {
        "name": "fraude-classifier",
        "namespace": namespace,
        "labels": {"opendatahub.io/dashboard": "true"},
        "annotations": {
            "openshift.io/display-name": "Classificador de fraude",
            "opendatahub.io/hardware-profile-name": "default-profile",
            "opendatahub.io/hardware-profile-namespace": "redhat-ods-applications",
            "security.opendatahub.io/enable-auth": "false",
            "serving.kserve.io/deploymentMode": "Standard",
            "serving.kserve.io/storageSecretName": "fraude-s3",
        },
    },
    "spec": {"predictor": {"minReplicas": 1, "maxReplicas": 1, "model": {
        "modelFormat": {"name": "sklearn", "version": "1"},
        "runtime": "fraude-sklearn",
        "protocolVersion": "v1",
        "storageUri": "s3://modelos/fraude",
        "resources": {"requests": {"cpu": "200m", "memory": "1Gi"}, "limits": {"cpu": "1", "memory": "4Gi"}},
    }}},
}

def pedir(metodo, url, corpo=None):
    data = json.dumps(corpo).encode() if corpo is not None else None
    req = urllib.request.Request(base + url, data=data, headers=headers, method=metodo)
    with urllib.request.urlopen(req, context=contexto, timeout=30) as resp:
        bruto = resp.read()
        return json.loads(bruto) if bruto else {}

try:
    atual = pedir("GET", caminho)
    manifesto["metadata"]["resourceVersion"] = atual["metadata"]["resourceVersion"]
    pedir("PUT", caminho, manifesto)
except urllib.error.HTTPError as erro:
    if erro.code != 404:
        raise
    pedir("POST", f"/apis/serving.kserve.io/v1beta1/namespaces/{namespace}/inferenceservices", manifesto)

for _ in range(30):
    status = pedir("GET", caminho).get("status", {})
    pronto = any(c.get("type") == "Ready" and c.get("status") == "True" for c in status.get("conditions", []))
    print(status.get("url"), pronto)
    if pronto:
        break
    time.sleep(10)


In [ ]:
req = urllib.request.Request(
    "https://inference-gateway.apps.ocp.p9cjv.sandbox2112.opentlc.com/v1/models/fraude-classifier:predict",
    data=json.dumps({"instances": [{"produto": "cartao", "valor": 8900}, {"produto": "pix", "valor": 40}]}).encode(),
    headers={"Content-Type": "application/json", "X-Model": "fraude-classifier"},
    method="POST",
)
with urllib.request.urlopen(req, context=ssl._create_unverified_context(), timeout=30) as resp:
    print(resp.status, resp.read().decode())
